# Notebook adaptado referente ao survey : https://arxiv.org/pdf/2005.00928 

O Notebook contem adaptações feitas por **Rafael Rezende, Augusto Mello e Victor Chaves** ,a fim de que fossem estudados e testados diferentes comportamentos para o modelo gpt2 (sendo que o paper originalmente se ocupava em estudar esses fenômenos de modo mais atrelado ao BERT). Nesse sentido, por se tratar de um modelo decoder, pôde-se não só visualizar a relação entre as atenções das palavras, mas também gerar próximas palavras das sentenças usadas como exemplo. Sendo assim, pôde-se observar, por meio de mapas de calor e de grafos, as seguintes técnicas propostas pelo paper serem colocadas em prática:


- **Raw Attention** = Método que usa somente da Self-Attention, sem considerar conexões residuais, fazendo com que as palavras percam seus sentidos próprios (sem conisederar o contexto) ao longo das camadas.

- **Attention Rollout** = *"Attention rollout and attention flow recursively compute the token attentions in each layer of a given model given the embedding attentions as input. The Attention Rollout technique, a chain of cumulative attention matrices is formed by multiplying the attention matrix 𝐴𝑙 of the l-th layer by the attention matrices of the subsequent layers".*

- **Attention Flow** = *"In attention flow, the weight of a single path is the minimum value of the weights of the edges in the path, instead of the product of the weights. Besides, we can not compute the attention for node s to node t by adding up the weights of all paths between these two nodes, since there might be an overlap between the paths and this might result in overflow in the overlapping edges".*


## Configuração Inicial e Bibliotecas:

### **Bibliotecas usadas** 

- **Pytorch** : Usada para diferentes fins, tais como gerar tensores necessários para alimentar o modelo, juntar esses tensores, além de ser usada para a aplicação de Softmax na camada dos logits de saída do modelo e para extrair o índice associado ao maior valor dentre essas probabilidades obtidas(para prever próximas palavras). 

- **Seaborn** : Ferramenta de alto nível que auxilia no desenho , faz os cálculos estatísticos, mapeia os valores numéricos para a paleta de cores e desenha. No final, o Seaborn não renderiza a imagem, ele devolve um objeto ax (um Axes do Matplotlib) com o "desenho" pronto.

- **Matplotlib** : Usado para gerar e salvar os gráficos necessários referente aos trés métodos de interpretabilidade por atenção estudados.

- **Transformers** : Dependência usada para importar o modelo decoder a ser utilizado, que, no caso, trata-se do Chat GPT2 versão Small, com 12 layers, 12 attention heads e uma dimensão de 768 dos embeddings , chamando também o tipo de treinamento que será usado (no caso é o de Causal Language Modeling que se relaciona com a predição do próximo token na sentença). Além disso, é chamado também o tipo de Tokenizador (AutoTokenizer), o qual está intimamente ligado ao algoritmo de Byte-Pair-Encoding (BPE).

In [ ]:
# import tensorflow_datasets as tfds
from attention_graph_util import *
import seaborn as sns
import matplotlib as mpl

import torch
import matplotlib.pyplot as plt
rc={'font.size': 10, 'axes.labelsize': 10, 'legend.fontsize': 10.0, 
    'axes.titlesize': 32, 'xtick.labelsize': 20, 'ytick.labelsize': 16}
plt.rcParams.update(**rc)
mpl.rcParams['axes.linewidth'] = .5 #set the value globally


from transformers import (
    AutoModelForCausalLM, AutoTokenizer
)


# Função de criação dos Heatmaps

- **Torch.flip**: recorta apenas as fatias de atenção onde o seu token de interesse (s_position) olha para os outros tokens da frase (t_positions). O flip inverte o eixo das camadas para uma questão de lógica visual: garante que a Camada 1 fique na base do gráfico e a última camada fique no topo.

- **cls_att[:, s_position] = 0.0**: zera a "auto-atenção" (o último token da sentença olhando para si mesmo). Isso foi feito a fim de que se evitasse um enviesamento na ultima coluna do gráfico, já que ela tenderia a ficar mais avermelhada sempre (útlimo token tende a acumular todo o contexto da frase , assim, ele só precisaria olhar para si mesmo ao longo da evolução das camadas).

- Como a biblioteca seaborn foi feita para trabalhar com matrizes do NumPy (e não tensores do PyTorch que podem estar na GPU), ela puxa a matriz de volta para a CPU e a converte para NumPy no exato momento da plotagem.


In [ ]:
def plot_attention_heatmap(att, s_position, t_positions, tokens_list):
    # Slice usando PyTorch
    cls_att = torch.flip(att[:, s_position, t_positions], dims=[0]).clone()
    
    cls_att[:, s_position] = 0.0 
    
    xticklb = [tokens_list[i] for i in t_positions]
    yticklb = [str(i) if i % 2 == 0 else '' for i in torch.arange(att.shape[0], 0, -1)]
    
    ax = sns.heatmap(cls_att.cpu().numpy(), xticklabels=xticklb, yticklabels=yticklb, cmap="YlOrRd")
    return ax

# Função de conversão de matriz 2D para 3D

 - Essa função faz o caminho reverso do processamento de grafos: ela pega uma malha grande 2D (a Matriz de Adjacência usada para calcular o fluxo) e a fatia de volta para o formato padrão 3D do PyTorch.

 - **mats = torch.zeros((n_layers, l, l))**: Cria um tensor 3D vazio pronto para receber os dados de cada camada individualmente (onde l é o número de tokens).  

 - **adjmat[(i+1)*l:(i+2)*l, i*l:(i+1)*l]**: Como a matriz grande agrupa os nós de L em L (onde L é o tamanho da frase), essa matemática de índices vai na matriz grande e recorta exatamente o quadrado L X L que contém o fluxo de informação cruzando da Camada i+1 de volta para a Camada i.

- Ela devolve um tensor limpo e organizado, pronto para escolher uma camada específica e passar para a função de heatmap do Seaborn.

In [ ]:
def convert_adjmat_tomats(adjmat, n_layers, l):
    mats = torch.zeros((n_layers, l, l))
    
    for i in range(n_layers):
        mats[i] = adjmat[(i+1)*l:(i+2)*l, i*l:(i+1)*l]
        
    return mats

## Carregando o Modelo e Escolhendo um Exemplo

- Nesse trecho, há a chamada de pesos pré-treinados do gpt2 além do uso do que foi importado na célula inicial para a instanciação do modelo de predição. 

- Além disso também é chamado o tokenizador que gera os tokens a partir da sentença de exemplo selecionada. **Uma ressalva relevante é que foi estudado e percebido que, por padrão arquitetural, os tokens de início do gpt2 tendem a armazenar muito mais atenção para si, tanto pelo fato de "puxarem" atenção de termos mais neutros da frase que não tem onde "jogar suas atenções", quanto devido a serem mais observados ao longo do processo de inferência (já que se evita o lookahead bias em casual language modeling)**. Para tanto, foi inserido um token de início [tokenizer.eos_token_id] de modo forçado, a fim de que ele puxasse essa atenção descartável para si, e que foi abandonado na visualização dos gráficos posteriormente.

- Após isso, forma-se um tensor em **input_ids = torch.tensor([tf_input_ids])** com esses tokens para alimentar o modelo posteriormente.

In [ ]:
pretrained_weights = 'gpt2'
model_id = pretrained_weights.split("/")[-1]
family = 'gpt'
print(f"Model: {model_id}, Family: {family}")

model = AutoModelForCausalLM.from_pretrained(
    pretrained_weights,
    output_hidden_states=True,
    output_attentions=True
)
model.zero_grad()
tokenizer = AutoTokenizer.from_pretrained(pretrained_weights, use_fast=True)

# Vamos analisar uma única frase como caso de estudo
sentence = "He talked to her about his book"
print(f"\nFrase em análise: '{sentence}'")

tokens =  tokenizer.tokenize(sentence)

# tokeniza a sentença
tf_input_ids = [tokenizer.eos_token_id] + tokenizer.encode(sentence)
tokens = [tokenizer.eos_token] + tokenizer.tokenize(sentence)

# transforma em um batch de sentença única
input_ids = torch.tensor([tf_input_ids])


## Inferência e Extração das Atenções (Raw Attention)


- A variável **all_attentions** retorna uma Tupla com 12 elementos (um elemento para cada camada do Transformer).Dentro de cada um desses 12 elementos, há um Tensor do PyTorch com o formato: (batch_size, num_heads, tamanho_da_frase, tamanho_da_frase). No nosso caso com uma única frase: (1, 12, L, L). Os valores preenchidos dentro dessa malha $L \times L$ são rigorosamente os resultados finais da equação da atenção: $Softmax(\frac{Q \times K^T}{\sqrt{d_k}})$. Eles representam a porcentagem exata de atenção que cada palavra prestou nas outras naquela cabeça específica

- Em  **_attentions = [att.detach() for att in all_attentions] e attentions_mat = torch.stack(_attentions)[:, 0]** pega-se aquela tupla e a converte em um único "super-cubo" 4D .O [:, 0] "mata" a dimensão do batch_size (já que só temos 1 frase), e a variável attentions_mat nasce com o formato final:(12, 12, L, L).12 (Eixo 0): Camadas 12 (Eixo 1): Cabeças de Atenção L x L (Eixos 2 e 3): A matriz de atenção com as conexões entre as palavras.

- Em **tokens = tokens[1:]  tf_input_ids = tf_input_ids[1:]** retira-se o token inserido inicialmente como sumidouro para o viés de atenção causal para não poluir o eixo x dos gráficos.

- Em **raw_att_norm**: Executa a heurística matemática de soma do Eixo 1 (que guarda as 12 Cabeças de Atenção) e divisão pelo total de cabeças (12). O resultado é uma matriz que representa a "Voz Média" da atenção em cada camada.

- Logo após, é feita uma plotagem do mapa de calor baseado na raw attention média. **O s_position=len(tf_input_ids) - 1** crava a análise na última palavra da frase, perguntando ao gráfico: "De onde veio a informação que formou esse último token nas diferentes camadas?".

In [ ]:
model_outputs = model(input_ids=input_ids)

# Extrai as atenções
all_attentions = model_outputs['attentions'] 
_attentions = [att.detach() for att in all_attentions] 
attentions_mat = torch.stack(_attentions)[:, 0]  # Remove dimensão do batch

# remove o token eos
tokens = tokens[1:] 
tf_input_ids = tf_input_ids[1:]

print(f"Formato final da matriz de atenção (Camadas, Cabeças, Seq, Seq): {attentions_mat.shape}")

# Predição do próximo token
output = model_outputs.logits[0]
predicted_target = torch.nn.Softmax(dim=0)(output[-1, :])
previewd = torch.argmax(predicted_target, dim=-1)

print(f"Próximo token adivinhado: {tokenizer.decode(previewd.item())}")

# Plot da Raw Attention Média (focando na última palavra)
plt.figure(figsize=(3,6))
t_pos = tuple(range(len(tokens)))

raw_att_norm = attentions_mat.sum(dim=1) / attentions_mat.shape[1]

plot_attention_heatmap(
    raw_att_norm,
    s_position=len(tf_input_ids) - 1,
    t_positions=t_pos,
    tokens_list=tokens
)

plt.title("Raw Attention Média")
plt.show()

## O Grafo e o Attention Flow

In [ ]:
# Preparação da matriz (Conexões residuais)
res_att_mat = attentions_mat.sum(dim=1) / attentions_mat.shape[1]

res_att_mat = res_att_mat + 0.5 * torch.eye(res_att_mat.shape[1])[None, ...]
res_att_mat = res_att_mat / res_att_mat.sum(dim=-1)[..., None]

# Construção do Grafo
res_adj_mat, res_labels_to_index = get_adjmat(mat=res_att_mat, input_tokens=tokens)

res_G = draw_attention_graph(
    res_adj_mat,
    res_labels_to_index,
    n_layers=res_att_mat.shape[0],
    length=res_att_mat.shape[-1]
)

plt.title("Grafo de Atenção Base")
plt.show()

# Calculando o Attention Flow
last_layer_name = f'L{attentions_mat.shape[0]}'
output_nodes = [key for key in res_labels_to_index if last_layer_name in key]
input_nodes = [key for key, idx in res_labels_to_index.items() if idx < attentions_mat.shape[-1]]

flow_values = compute_flows(res_G, res_labels_to_index, input_nodes, length=attentions_mat.shape[-1])

flow_att_mat = convert_adjmat_tomats(
    flow_values,
    n_layers=attentions_mat.shape[0],
    l=attentions_mat.shape[-1]
)

plt.figure(figsize=(3,6))
plot_attention_heatmap(
    flow_att_mat,
    s_position=len(tf_input_ids) - 1,
    t_positions=t_pos,
    tokens_list=tokens
)

plt.title("Attention Flow")
plt.show()

## Attention Rollout

In [ ]:
# CÉLULA 5
joint_attentions = compute_joint_attention(res_att_mat, add_residual=False)

plt.figure(figsize=(3,6))
plot_attention_heatmap(joint_attentions, s_position=len(tf_input_ids)-1, t_positions=t_pos, tokens_list=tokens)
plt.title("Attention Rollout")
plt.show()

## Loop com mais exemplos

In [ ]:
sentences2 = {}

sentences2[0] = "He talked to her about his book"
sentences2[1] = "She asked the doctor about her"
sentences2[2] = "The author talked to Sara about his"
sentences2[3] = "John tried to convince Mary of his love and brought flowers for "
sentences2[4] = "Mary convinced John of her"
sentences2[5] = "Barack Obama was the president of the"
sentences2[6] = "Artificial intelligence is the field of study that"
sentences2[7] = "Why is the sky blue?"
sentences2[8] = "the capital of France is"

for ex_id in range(len(sentences)):
    
    OUTPUT_DIR = IMAGES_DIR / str(ex_id)
    OUTPUT_DIR.mkdir(exist_ok=True)
    
    sentence = sentences[ex_id]

    tokens = tokenizer.tokenize(sentence)

    # tokeniza a sentença
    tf_input_ids = [tokenizer.eos_token_id] + tokenizer.encode(sentence)
    tokens = [tokenizer.eos_token] + tokenizer.tokenize(sentence)

    # transforma em batch
    input_ids = torch.tensor([tf_input_ids])

    # saída do modelo
    model_outputs = model(input_ids=input_ids)

    # hidden states e atenções
    all_hidden_states, all_attentions = model_outputs['hidden_states'], model_outputs['attentions']

    # transforma tudo em tensor PyTorch
    _attentions = [att.detach() for att in all_attentions]

    # 12 camadas, 1 batch, 12 cabeças, LXL
    attentions_mat = torch.stack(_attentions)[:, 0]

    # remove o token eos
    attentions_mat = attentions_mat[:, :, 1:, 1:]
    tokens = tokens[1:]
    tf_input_ids = tf_input_ids[1:]

    print(input_ids)
    print(tokens)

    # =========================
    # geração de texto
    # =========================
    max_l = 30
    output_ids = model.generate(input_ids, max_length=max_l)

    print(f"Próximos {max_l} tokens gerados pelo modelo:\n{tokenizer.decode(output_ids[0], skip_special_tokens=True)}")

    # logits
    output = model(input_ids).logits[0]

    predicted_target = torch.nn.Softmax(dim=0)(output[-1, :])
    previewd = torch.argmax(predicted_target, dim=-1)

    print(f"Próximo token gerado pelo modelo:\n{tokenizer.decode(previewd.item())}")

    # =========================
    # Top-k
    # =========================
    k = 5
    topk_vals, topk_idx = torch.topk(predicted_target, k)

    print(f"Top {k} tokens:")
    for idx, val in zip(topk_idx, topk_vals):
        print(f"Token: {tokenizer.decode([idx.item()]):15} | prob: {val.item():.6f} | id: {idx.item()}")

    print("\n\n")

    # =========================
    # Barplot
    # =========================
    yax = [predicted_target[id].item() for id in topk_idx]
    xax = [tokenizer.decode([id.item()]) for id in topk_idx]

    fig = plt.figure(figsize=(6,6))
    ax = sns.barplot(x=xax, y=yax, linewidth=0)

    sns.despine(fig=fig, ax=None, top=True, right=True, left=True, bottom=False)

    ax.set_ylim(0,1)

    plt.savefig(OUTPUT_DIR / f'rat_gpt_bar_{ex_id}.png', format='png', transparent=True, dpi=360, bbox_inches='tight')
    plt.close()

    # =========================
    # Raw Attention
    # =========================
    plt.figure(figsize=(3,6))

    t_pos = tuple(range(len(tokens)))

    raw_att = attentions_mat.sum(dim=1) / attentions_mat.shape[1]

    plot_attention_heatmap(
        raw_att,
        s_position=len(tf_input_ids)-1,
        t_positions=t_pos,
        tokens_list=tokens
    )

    plt.savefig(OUTPUT_DIR / f'rat_gpt_att_{ex_id}.png', format='png', transparent=True, dpi=360, bbox_inches='tight')
    plt.close()

    # =========================
    # Residual Attention
    # =========================
    res_att_mat = attentions_mat.sum(dim=1) / attentions_mat.shape[1]

    res_att_mat = res_att_mat + torch.eye(res_att_mat.shape[1])[None, ...]
    res_att_mat = res_att_mat / res_att_mat.sum(dim=-1)[..., None]

    res_adj_mat, res_labels_to_index = get_adjmat(mat=res_att_mat, input_tokens=tokens)

    res_G = draw_attention_graph(
        res_adj_mat,
        res_labels_to_index,
        n_layers=res_att_mat.shape[0],
        length=res_att_mat.shape[-1]
    )

    # =========================
    # Attention Flow
    # =========================
    last_layer_name = f'L{attentions_mat.shape[0]}'

    output_nodes = []
    input_nodes = []

    for key in res_labels_to_index:
        if last_layer_name in key:
            output_nodes.append(key)
        if res_labels_to_index[key] < attentions_mat.shape[-1]:
            input_nodes.append(key)

    flow_values = compute_flows(res_G, res_labels_to_index, input_nodes, length=attentions_mat.shape[-1])

    flow_att_mat = convert_adjmat_tomats(
        flow_values,
        n_layers=attentions_mat.shape[0],
        l=attentions_mat.shape[-1]
    )

    plt.figure(figsize=(3,6))

    plot_attention_heatmap(
        flow_att_mat,
        s_position=len(tf_input_ids)-1,
        t_positions=t_pos,
        tokens_list=tokens
    )

    plt.savefig(OUTPUT_DIR / f'res_fat_gpt_att_{ex_id}.png', format='png', transparent=True, dpi=360, bbox_inches='tight')
    plt.close()

    # =========================
    # Attention Rollout
    # =========================
    joint_attentions = compute_joint_attention(res_att_mat, add_residual=False)

    plt.figure(figsize=(3,6))

    plot_attention_heatmap(
        joint_attentions,
        s_position=len(tf_input_ids)-1,
        t_positions=t_pos,
        tokens_list=tokens
    )

    plt.savefig(OUTPUT_DIR / f'res_jat_gpt_att_{ex_id}.png', format='png', transparent=True, dpi=360, bbox_inches='tight')
    plt.close()